# Create Dataset for Build Repairs

## Setup

In [ ]:
import weave

PROJECT_NAME = "bugbug-build-repair-eval"
DATASET_NAME = "build_repair_one_commit_eval"

_ = weave.init(PROJECT_NAME)

## Prepare the Data

### Load one commit build failures

In [1]:
import pandas as pd

df = pd.read_json(
    "https://community-tc.services.mozilla.com/api/queue/v1/task/Ra5r2qSyS8G-9pjLrS6l6Q/runs/0/artifacts/public%2Fci_failures.json.zst",
    lines=True,
)
df = df[(df.failure_commits.apply(len) == 1) & (df.fix_commits.apply(len) == 1)]
print(f"One commit fail and fix: {len(df)}")
df = df[
    df.failures.apply(
        lambda fails: any(
            "build" in f["task_name"] and "test" not in f["task_name"] for f in fails
        )
    )
]
print(f"Build fails only: {len(df)}")

One commit fail and fix: 388
Build fails only: 89


### Check that corresponding bugs are public

In [4]:
import requests


def is_bug_public(bug_id):
    url = f"https://bugzilla.mozilla.org/rest/bug/{bug_id}"
    resp = requests.get(url)
    if resp.status_code == 401 or resp.status_code == 504:
        return False
    elif resp.status_code == 200:
        return True
    else:
        raise ValueError(f"Unexpected Bugzilla status code: {resp.status_code}")


df = df[df.bug_id.apply(is_bug_public)]
print(f"Build fails for public bugs: {len(df)}")

KeyboardInterrupt: 

### Get GitHub revisions

In [ ]:
def get_git_rev(hg_revs):
    for rev in hg_revs:
        convert_url = f"https://lando.moz.tools/api/hg2git/firefox/{rev}"
        resp = requests.get(convert_url)
        if resp.status_code != 200:
            raise ValueError(f"Unexpected HTTP status code: {resp.status_code}. {resp}")
        yield resp.json()["git_hash"]


df["gh_failure_commits"] = df.failure_commits.apply(
    lambda commits: list(get_git_rev(commits))
)
df["gh_fix_commits"] = df.fix_commits.apply(lambda commits: list(get_git_rev(commits)))
df = df.rename(
    columns={"failure_commits": "hg_failure_commits", "fix_commits": "hg_fix_commits"}
)

### Ger bugzilla comments before the fix

In [ ]:
from libmozdata import config

from bugbug.tools.core.platforms.bugzilla import Bug

config.set_default_value("User-Agent", "name", "bugbug/1.0")


def _get_comments(bug, fail_commit):
    for comment in bug._metadata["comments"]:
        if comment["creator"] == "pulsebot@bmo.tld":
            if fail_commit[:12] in comment["raw_text"]:
                # stop adding comments at failure commit push
                yield comment["raw_text"]
                break
            else:
                continue
        if comment["raw_text"]:
            yield comment["raw_text"]


def get_bug_info_pre_fix(build_fail):
    bug_id = build_fail["bug_id"]
    fail_commit = build_fail["hg_failure_commits"][0]

    try:
        bug = Bug.get(bug_id)
    except ValueError as ex:
        print(ex)
        return None

    return {"title": bug.summary, "comments": list(_get_comments(bug, fail_commit))}


df["pre_fix_bug"] = df.apply(get_bug_info_pre_fix, axis=1)
df = df[df["pre_fix_bug"].notnull()]
print(f"Final number of bugs: {len(df)}")

In [ ]:
df

## Save the Dataset

In [ ]:
examples = df.to_dict(orient="records")
examples[0]

In [ ]:
dataset = weave.Dataset(
    name=DATASET_NAME,
    description="Build repair evaluation dataset with failure logs, ground truth fix commits and pre fix Bugzilla comments.",
    rows=examples,
)

_ = weave.publish(dataset)